
#  OpenMP GPU入門(4) --- GPU上での台数効果の測定
# 1. おさらい
* ここまでで, `target` / `teams` / `distribute` / `parallel` / `for` (`do`) を使って, ループをGPU上の多数のスレッドで分割実行できるようになった
* このトピックでは, CPUのときと同様, スレッド数(さらにチーム数)を増やすとGPU上で実際にどれだけ速くなるか(<font color="blue">台数効果</font>, スピードアップ)を測定する
* 性能の指標として GFLOPS (1秒あたり何十億回の浮動小数点演算ができたか) を用いる

# 2. 環境設定
* これまでのトピックと同様, コンパイラとジョブ投入の設定を行う
* カーネルを再スタートしたときなどは失われるのでそのたびに行うこと

In [ ]:
import os
paths = os.environ["PATH"].split(":")
nvc_path = "/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin"
fj_path = "/opt/FJSVxtclanga/tcsds-1.2.41/bin"
for path in [nvc_path, fj_path]:
    if path not in paths:
        paths = [path] + paths
os.environ["PATH"] = ":".join(paths)

In [ ]:
%%bash
which nvc++
which nvfortran

In [ ]:
import wisteria_submit

In [ ]:
import heytutor

# 3. 台数効果とは (GPU編)
* GPUでは, $T$個のチーム × $H$個のスレッド = $T \times H$ 個のスレッドが並列に動く
* CPUのときと同じように, 並列度(スレッド数・チーム数)を増やしても, それに比例して速くなるとは限らない
* GPUの場合, 1個のチーム(Streaming Multiprocessor, SM上で実行される)が抱えるスレッド数や, 全体のチーム数が少なすぎると, GPUの演算器を十分に使い切れず, 性能が出ない
* 逆にハードウェアの並列度を超えてチーム数・スレッド数を増やしても, それ以上は速くならない
* ここでは, 計算自身にあまり意味はないが簡単な例題で, GPU上での性能向上を目撃してみる
* `lin_rec` は `x = a * x + b` を$n$回繰り返す関数で, 1回につき乗算1回・加算1回, 計$2n$回の浮動小数点演算を行う
* それを$m$個の独立な要素について並列に計算する ( `#pragma omp target teams distribute parallel for` / `!$omp target teams distribute parallel do` )

```
OMP_NUM_TEAMS=nteams OMP_NUM_THREADS=nthreads ./omp_speedup_gpu.exe m n
```
* とすると, チーム数=`nteams`, スレッド数=`nthreads` で実行する
* `m`, `n` を省略すると, `m` = `nteams` $\times$ `nthreads`, `n` = $100 \times 1000 \times 1000$ とする

## 3-1. C++版

In [ ]:
%%writefile omp_speedup.cpp
#include <cassert>
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* x = ax + b をひたすら n 回繰り返す.
   (|a| < 1.0 なら c によらず, x = b / (1 - a) に収束).
   n 回 mul + add を行う (-> 2 n flops) */
double lin_rec(double a, double b, double c, long n) {
  double x = c;
  for (long j = 0; j < n; j++) {
    x = a * x + b;
  }
  return x;
}

int main(int argc, char ** argv) {
  /* チーム数・スレッド数を環境変数から取得 */
  char * nteams_   = getenv("OMP_NUM_TEAMS");
  int    nteams    = (nteams_   ? atoi(nteams_)   : 1);
  char * nthreads_ = getenv("OMP_NUM_THREADS");
  int    nthreads  = (nthreads_ ? atoi(nthreads_) : 1);
  long m = (1 < argc ? atol(argv[1]) : (long)nteams * nthreads);
  long n = (2 < argc ? atol(argv[2]) : 100 * 1000 * 1000);
  double * x = (double *)calloc(sizeof(double), m);
  assert(x);
  printf("num_teams = %d, num_threads = %d\n", nteams, nthreads);
  printf("m = %ld, n = %ld\n", m, n);
  /* 計測開始 */
  double t0 = omp_get_wtime();
  /* 計算本体 (GPU にオフロード) */
#pragma omp target teams distribute parallel for num_teams(nteams) num_threads(nthreads) map(tofrom: x[0:m])
  for (long i = 0; i < m; i++) {
    x[i] = lin_rec(0.99, i + 1, 1.0, n);
  }
  /* 計測終了 */
  double t1 = omp_get_wtime();
  double dt = t1 - t0;          /* sec */
  /* 答え確認 (x[i] = 100 * (i + 1) くらいのはず) */
  long err = 0;
  for (long i = 0; i < m; i++) {
    if (fabs(x[i] - 100 * (i + 1)) > 1.0e-3) {
      printf("x[%3ld] = %9.3f\n", i, x[i]);
      err++;
    }
  }
  if (err == 0) printf("OK\n");
  double flops = 2. * (double)m * (double)n;
  printf("elapsed    : %7.3f  sec\n", dt);
  printf("elapsed/m  : %7.3f msec\n", dt / m * 1e3);
  printf("elapsed/n  : %7.3f nsec\n", dt / n * 1e9);
  printf("elapsed/mn : %7.3f nsec\n", dt / (m * n) * 1e9);
  printf("flops      : %.2e\n", flops);
  printf("%.3f GFLOPS\n", flops / dt * 1e-9);
  return 0;
}

In [ ]:
%%bash
nvc++ -fast -mp=gpu omp_speedup.cpp -o omp_speedup_gpu.exe

## 3-2. Fortran版

In [ ]:
%%writefile omp_speedup.f90
module lin_rec_mod
contains
  ! x = ax + b をひたすら n 回繰り返す.
  ! (|a| < 1.0 なら c によらず, x = b / (1 - a) に収束).
  ! n 回 mul + add を行う (-> 2 n flops)
  !$omp declare target
  function lin_rec(a, b, c, n) result(x)
    real(8), intent(in) :: a, b, c
    integer(8), intent(in) :: n
    real(8) :: x
    integer(8) :: j
    x = c
    do j = 1, n
       x = a * x + b
    end do
  end function lin_rec
end module lin_rec_mod

program omp_speedup
  use omp_lib
  use lin_rec_mod
  character(len=32) :: arg, env
  integer(8) :: m, n, i
  integer :: nteams, nthreads, st
  integer(8) :: err
  real(8), allocatable :: x(:)
  real(8) :: t0, t1, dt, flops
  ! チーム数・スレッド数を環境変数から取得
  nteams = 1
  call get_environment_variable("OMP_NUM_TEAMS", env, status=st)
  if (st == 0) read (env, *) nteams
  nthreads = 1
  call get_environment_variable("OMP_NUM_THREADS", env, status=st)
  if (st == 0) read (env, *) nthreads
  m = int(nteams, 8) * int(nthreads, 8)
  n = 100 * 1000 * 1000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) m
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) n
  end if
  allocate(x(m))
  print "(a,i0,a,i0)", "num_teams = ", nteams, ", num_threads = ", nthreads
  print "(a,i0,a,i0)", "m = ", m, ", n = ", n
  ! 計測開始
  t0 = omp_get_wtime()
  ! 計算本体 (GPU にオフロード)
  !$omp target teams distribute parallel do num_teams(nteams) num_threads(nthreads) map(tofrom: x)
  do i = 1, m
     x(i) = lin_rec(0.99d0, real(i, 8), 1.0d0, n)
  end do
  !$omp end target teams distribute parallel do
  ! 計測終了
  t1 = omp_get_wtime()
  dt = t1 - t0                  ! sec
  ! 答え確認 (x(i) = 100 * i くらいのはず)
  err = 0
  do i = 1, m
     if (abs(x(i) - 100 * i) > 1.0d-3) then
        print "(a,i3,a,f9.3)", "x[", i, "] = ", x(i)
        err = err + 1
     end if
  end do
  if (err == 0) print "(a)", "OK"
  flops = 2.d0 * real(m, 8) * real(n, 8)
  print "(a,f7.3,a)", "elapsed    : ", dt, "  sec"
  print "(a,f7.3,a)", "elapsed/m  : ", dt / m * 1e3, " msec"
  print "(a,f7.3,a)", "elapsed/n  : ", dt / n * 1e9, " nsec"
  print "(a,f7.3,a)", "elapsed/mn : ", dt / (m * n) * 1e9, " nsec"
  print "(a,es9.2)",  "flops      : ", flops
  print "(f7.3,a)", flops / dt * 1e-9, " GFLOPS"
end program omp_speedup

In [ ]:
%%bash
nvfortran -fast -mp=gpu omp_speedup.f90 -o omp_speedup_f_gpu.exe

# 4. スレッド数を変えてみる
* GPUは計算ノードにのみ搭載されているので, GPUを使う実行は `%%bash_submit` でジョブとして投入する
* 必要に応じて rscgrp や elapse などを指定して実行せよ
* まず, `OMP_NUM_TEAMS=1` (チーム数1) に固定し, `OMP_NUM_THREADS` (スレッド数) だけを変えて性能向上を確認する
* <font color="red">`OMP_NUM_THREADS` は 1 でなければ, 32 の倍数でないといけないことに注意</font> (GPUのスレッドは32本単位(ワープ)で動くため)
* まずは1チーム・1スレッドで実行してみる

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

OMP_NUM_TEAMS=1 OMP_NUM_THREADS=1 ./omp_speedup_gpu.exe

* 手動でやるのが嫌になったら以下で一撃で実行 (スレッド数を 1, 32, 64, ... と増やしていく)

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

for th in 1 32 64 128 256 512 1024 ; do
    echo -n "$th "
    OMP_NUM_TEAMS=1 OMP_NUM_THREADS=${th} ./omp_speedup_gpu.exe | grep GFLOPS
done

* 結果を以下で可視化 (上の結果をコピペせよ)
* 直線が「理想的な台数効果(スレッド数に比例)」, もう一方が実測値

In [ ]:
import matplotlib.pyplot as plt

DATA = r"""
1 xxxx GFLOPS
2 xxxx GFLOPS
3 xxxx GFLOPS
4 xxxx GFLOPS
6 xxxx GFLOPS
8 xxxx GFLOPS
12 xxxx GFLOPS
16 xxxx GFLOPS
20 xxxx GFLOPS
24 xxxx GFLOPS
28 xxxx GFLOPS
32 xxxx GFLOPS
"""

def main():
    data = DATA.strip().split("\n")
    X = []
    Y = []
    L = []
    for line in data:
        fields = line.strip().split()
        if len(fields) != 3:
            continue
        x, y = fields[:2]
        X.append(float(x))
        Y.append(float(y))
        L.append(Y[0] / X[0] * float(x))
    plt.ylabel("GFLOPS")
    plt.xlabel("num_threads")
    plt.plot(X, L)
    plt.plot(X, Y)
    plt.show()

main()

# 5. チーム数を変えてみる
* 次に, スレッド数を上記で性能が頭打ちになった(あるいは最も良かった)値で固定した上で, `OMP_NUM_TEAMS` (チーム数) を色々変えて実行する
* まず固定した1チームでの基準を測る

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

OMP_NUM_TEAMS=1 OMP_NUM_THREADS=最適なスレッド数 ./omp_speedup_gpu.exe

* 手動でやるのが嫌になったら以下で一撃で実行 (チーム数 `tm` を増やしていく)
* このGPU (NVIDIA A100) は 108 個の Streaming Multiprocessor (= チームを割り当てられる単位) を備えているので, おおむね 108 までチーム数を増やすと性能が向上し, それ以降は頭打ちになるはず

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

th=最適なスレッド数
for tm in 1 2 4 8 16 32 64 108 ; do
    echo -n "$((tm * th)) "
    OMP_NUM_TEAMS=${tm} OMP_NUM_THREADS=${th} ./omp_speedup_gpu.exe | grep GFLOPS
done

* 結果を以下で可視化 (上の結果をコピペせよ)
* 横軸は (チーム数 × スレッド数), すなわち総スレッド数になっている

In [ ]:
import matplotlib.pyplot as plt

DATA = r"""
1 xxxx GFLOPS
2 xxxx GFLOPS
3 xxxx GFLOPS
4 xxxx GFLOPS
6 xxxx GFLOPS
8 xxxx GFLOPS
12 xxxx GFLOPS
16 xxxx GFLOPS
20 xxxx GFLOPS
24 xxxx GFLOPS
28 xxxx GFLOPS
32 xxxx GFLOPS
"""

def main():
    data = DATA.strip().split("\n")
    X = []
    Y = []
    L = []
    for line in data:
        fields = line.strip().split()
        if len(fields) != 3:
            continue
        x, y = fields[:2]
        X.append(float(x))
        Y.append(float(y))
        L.append(Y[0] / X[0] * float(x))
    plt.ylabel("GFLOPS")
    plt.xlabel("num_threads")
    plt.plot(X, L)
    plt.plot(X, Y)
    plt.show()

main()

# 6. まとめ
* GPUでは, 1チームあたりのスレッド数(`OMP_NUM_THREADS`)と, チーム数(`OMP_NUM_TEAMS`)の両方を十分に大きくして, 初めてGPUの性能を引き出せる
* `OMP_NUM_THREADS` は 1 または 32 の倍数でなければならない
* チーム数は GPU の Streaming Multiprocessor 数 (A100 では 108) 程度までは性能向上が期待できる
* なお, CPUのときと同様に, この実験はいろいろな意味でいい加減な計測であることに注意 (本来は複数回実行して平均をとる, 最初の1回はオフロードの初期化コストが余分にかかる, など)